In [ ]:
!pip install langchain

In [ ]:
!pip install -U transformers accelerate sentence-transformers faiss-cpu

In [ ]:
!pip install -U langchain langchain-community pypdf

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("/content/intro-to-ml.pdf")
documents = loader.load()

In [ ]:
print(f"Total Pages: {len(documents)}")
print(documents[0].page_content[:500])

Total Pages: 392
Andreas C. Müller & Sarah Guido
Introduction to 
Machine 
Learning  
with P y t h o n   
A GUIDE FOR DATA SCIENTISTS


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

print(f"Total chunks: {len(chunks)}")

Total chunks: 1137


In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/tmp/ipykernel_23696/384917042.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(chunks, embedding_model)
vectorstore.save_local("faiss_index")

In [ ]:
query = "What is supervised learning?"
docs = vectorstore.similarity_search(query, k=3)

for doc in docs:
    print(doc.page_content[:300])

CHAPTER 2
Supervised Learning
As we mentioned earlier, supervised machine learning is one of the most commonly
used and successful types of machine learning. In this chapter, we will describe super‐
vised learning in more detail and explain several popular supervised learning algo‐
rithms. We alread
2. Supervised Learning. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .  25
Classification and Regression                                                                                         25
Generalization, Overfitting, and Underfit
chapter, we will go into more depth about the different kinds of supervised models in
scikit-learn and how to apply them successfully.
24 | Chapter 1: Introduction


In [ ]:
def retrieve_docs(query, k=5):
    return vectorstore.similarity_search(query, k=k)

In [ ]:
from transformers import pipeline

llm = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM',

In [ ]:
def generate_answer(query, docs):
    context = "\n\n".join([d.page_content for d in docs])

    prompt = f"""
You are a Machine Learning expert.

Answer ONLY using the context below.
If the answer is not present, say "I don't know".

Context:
{context}

Question:
{query}

Answer:
"""

    result = llm(prompt, max_new_tokens=150)
    return result[0]['generated_text']

In [ ]:
def rag_pipeline(query):
    docs = vectorstore.similarity_search(query, k=5)
    return generate_answer(query, docs)

In [ ]:
while True:
    query = input("\nAsk a question: ")

    if query.lower() == "exit":
        break

    print(rag_pipeline(query))


Ask a question: What is machine learning


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



You are a Machine Learning expert.

Answer ONLY using the context below.
If the answer is not present, say "I don't know".

Context:
CHAPTER 1
Introduction
Machine learning is about extracting knowledge from data. It is a research field at the
intersection of statistics, artificial intelligence, and computer science and is also
known as predictive analytics or statistical learning. The application of machine
learning methods has in recent years become ubiquitous in everyday life. From auto‐
matic recommendations of which movies to watch, to what food to order or which
products to buy, to personalized online radio and recognizing your friends in your
photos, many modern websites and devices have machine learning algorithms at their
core. When you look at a complex website like Facebook, Amazon, or Netflix, it is
very likely that every part of the site contains multiple machine learning models.

very likely that every part of the site contains multiple machine learning models.
Outside o

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



You are a Machine Learning expert.

Answer ONLY using the context below.
If the answer is not present, say "I don't know".

Context:
CHAPTER 2
Supervised Learning
As we mentioned earlier, supervised machine learning is one of the most commonly
used and successful types of machine learning. In this chapter, we will describe super‐
vised learning in more detail and explain several popular supervised learning algo‐
rithms. We already saw an application of supervised machine learning in Chapter 1:
classifying iris flowers into several species using physical measurements of the
flowers.
Remember that supervised learning is used whenever we want to predict a certain
outcome from a given input, and we have examples of input/output pairs. We build a
machine learning model from these input/output pairs, which comprise our training
set. Our goal is to make accurate predictions for new, never-before-seen data. Super‐

chapter, we will go into more depth about the different kinds of supervised mo